## **ENTRENAMIENTO SAM 3 CON KVASIR-SEG**

En este fichero se evalúa el rendimiento del modelo SAM 3 tras haberlo entrenado usando el dataset KVASIR-SEG.

En este primer bloque de código se importan las librerías necesarias para la ejecución del fichero completo. Además, se importan las funciones de métricas y de medición de inferencia. Para ello se tuvo que añadir la raíz del proyecto al path de Python para que estos imports funcionen correctamente. Finalmente, se define la ruta al datset con el que entrenaremos los modelos.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from ultralytics import SAM
from ultralytics.models.sam import SAM3Predictor
import numpy as np
import os
import cv2
import pandas as pd
import json
import time
import sys
import torch
import shutil
import random
import torch.nn as nn


root_path = os.path.abspath('..')
if root_path not in sys.path:
    sys.path.append(root_path)

from utils.segmentation_quality_metrics import boundary_iou, hausdorff_95, compute_all_metrics
from utils.efficiency_metrics import measure_inference_central_point

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG"


La siguiente función divide el conjunto de datos de imágenes en entrenamiento, validación y test (70, 15, 15) creando las subcarpetas correspondientes tanto para imágenes como para máscaras, garantizando la correspondencia entre ambas. Para ello, se establece una semilla, evitando así crear conjuntos diferentes cada vez que se ejecute la función. 

In [ ]:
def split_dataset(dataset, train_ratio=0.7, val_ratio=0.15):
    images_dir = os.path.join(dataset, "images")
    masks_dir = os.path.join(dataset, "masks")

    images = [f.replace(".jpg", "") for f in os.listdir(images_dir) if f.endswith(".jpg")]
    random.seed(42)
    random.shuffle(images)

    n = len(images)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    splits = {
        "train": images[:n_train],
        "val": images[n_train:n_train + n_val],
        "test": images[n_train+n_val:]
    }

    for split, names in splits.items():
        os.makedirs(os.path.join(dataset, "images", split), exist_ok=True)
        os.makedirs(os.path.join(dataset, "masks",  split), exist_ok=True)
        for name in names:
            shutil.copy(os.path.join(images_dir, name + ".jpg"), os.path.join(dataset, "images", split, name + ".jpg"))
            shutil.copy(os.path.join(masks_dir,  name + ".jpg"), os.path.join(dataset, "masks",  split, name + ".jpg"))

split_dataset(dataset)


En primer lugar, se habilita TF32 para las operaciones matriciales, ya que esto nos permite acelerar el entrenamiento en GPUs NVIDIA modernas con una pérdida de precisión mínima. Además, se define el dispositivo donde se ejecutará el entrenamiento y se comprueba la disponibilidad de CUDA.

A continuación, se define el dataset cargando en el constructor las rutas de las imágenes y máscaras del split correspondiente (train, val o test), junto con el fichero json que contiene las cajas delimitadoras de cada imagen.

Por último, en el método __getitem__ se aplica el preprocesamiento de cada muestra. La imagen se redimensiona a 1008x1008 (ya que tenía que ser múltiplo de 14), se normaliza entre 0 y 1 y se convierte a tensor. Por otro lado, la máscara se redimensiona a 288x288 (resolución de salida del mask decoder de SAM 3) y se binariza. Finalmente, a partir de la caja delimitadora se calcula el punto central escalado a las nuevas dimensiones de la imagen, que es el prompt que se le pasaría al modelo.

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

class SegDataset(Dataset):
    def __init__(self, dataset_path, split, bbox_json, img_size=1008):
        self.img_size   = img_size
        self.images_dir = os.path.join(dataset_path, "images", split)
        self.masks_dir  = os.path.join(dataset_path, "masks",  split)
        self.samples    = []

        with open(bbox_json) as f:
            bbox_json = json.load(f)

        for img_name, info in bbox_json.items():
            img_path  = os.path.join(self.images_dir, img_name + ".jpg")
            mask_path = os.path.join(self.masks_dir,  img_name + ".jpg")
            if os.path.exists(img_path) and os.path.exists(mask_path):
                self.samples.append((img_path, mask_path, info["bbox"][0]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path, bbox = self.samples[idx]

        image = cv2.imread(img_path)
        orig_h, orig_w = image.shape[:2]
        scale_x = self.img_size / orig_w
        scale_y = self.img_size / orig_h
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.img_size, self.img_size))
        image = torch.tensor(image).permute(2, 0, 1).float() / 255.0

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (288, 288))
        mask = torch.tensor((mask > 127).astype(np.float32)).unsqueeze(0)

        xmin, ymin, xmax, ymax = bbox["xmin"], bbox["ymin"], bbox["xmax"], bbox["ymax"]
        point = torch.tensor([[(xmin + (xmax-xmin)/2) * scale_x, (ymin + (ymax-ymin)/2) * scale_y]]).float()
        label = torch.tensor([1])

        return image, mask, point, label


Esta función contiene el bucle de entrenamiento. En primer lugar, se carga el modelo mediante el wrapper de Ultralytics SAM, del que se extrae el modelo interno con .model. Al igual que con los modelos anteriores, se congelan tanto el image encoder como el prompt encoder, de forma que durante el entrenamiento solo se actualizan los pesos del mask decoder.

Como optimizador se usa Adam con un learning rate de 1e-4, y se añade un scheduler que reduce el learning rate a la mitad cada 20 épocas, ayudando así a afinar el modelo conforme avanza el entrenamiento. Además, se usa BCEWithLogitsLoss como función de pérdida, ya que es adecuada para segmentación binaria.

Por último, al no poder usar predictor.set_image directamente, el image encoder se ejecuta manualmente mediante forward_image y _prepare_backbone_features, configurando explícitamente los tamaños de características esperados por el decoder. El resto del proceso es equivalente al de SAM 2, ya que el prompt encoder y el image encoder se ejecutan sin calcular gradientes, el mask decoder sí los calcula, la pérdida se promedia sobre el batch y los pesos se guardan en formato de diccionario con la clave "model".

In [ ]:
def train_sam(dataset_path, bbox_json, weights_path, output_weights, epochs=50, batch_size=4):
    sam3_wrapper = SAM(weights_path)
    sam3 = sam3_wrapper.model
    for param in sam3.parameters():
        param.data = param.data.to(device)
    for buffer in sam3.buffers():
        buffer.data = buffer.data.to(device)
    sam3.to(device)

    for param in sam3.image_encoder.parameters():
        param.requires_grad = False
    for param in sam3.sam_prompt_encoder.parameters():
        param.requires_grad = False

    optimizer = torch.optim.Adam(sam3.sam_mask_decoder.parameters(), lr=1e-4)
    scheduler  = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
    loss_fn   = nn.BCEWithLogitsLoss()

    train_dataset = SegDataset(dataset_path, "train", bbox_json)
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    predictor = SAM3Predictor(overrides={"model": weights_path, "task": "segment", "mode": "predict", "verbose": False})
    predictor.setup_model(sam3)

    predictor._bb_feat_sizes = [(288, 288), (144, 144), (72, 72)]
    
    sam3.train()

    for epoch in range(epochs):
        total_loss = 0
        for images, masks, points, labels in train_loader:
            masks  = masks.to(device)
            points = points.to(device)
            labels = labels.to(device)

            loss_batch = 0
            for i in range(images.shape[0]):

                with torch.no_grad():
                    image_tensor = images[i].unsqueeze(0).to(device)
                    backbone_out = sam3.forward_image(image_tensor)
                    _, vision_feats, _, _ = sam3._prepare_backbone_features(backbone_out)
                    #print(len(vision_feats), [v.shape for v in vision_feats])
                    if sam3.directly_add_no_mem_embed:
                        vision_feats[-1] = vision_feats[-1] + sam3.no_mem_embed
                    feats = [
                        feat.permute(1, 2, 0).reshape(1, -1, *feat_size)
                        for feat, feat_size in zip(vision_feats, predictor._bb_feat_sizes)
                    ]
                    features = {"image_embed": feats[-1], "high_res_feats": feats[:-1]}

                with torch.no_grad():
                    sparse_embeddings, dense_embeddings = sam3.sam_prompt_encoder(
                        points=(points[i].unsqueeze(0), labels[i].unsqueeze(0)),
                        boxes=None,
                        masks=None
                    )

                image_embedding = features["image_embed"]
                image_pe        = sam3.sam_prompt_encoder.get_dense_pe()
                high_res_features = features["high_res_feats"]

                low_res_masks, _, _, _ = sam3.sam_mask_decoder(
                    image_embeddings=image_embedding,
                    image_pe=image_pe,
                    sparse_prompt_embeddings=sparse_embeddings,
                    dense_prompt_embeddings=dense_embeddings,
                    multimask_output=False,
                    repeat_image=False,
                    high_res_features=high_res_features,
                )

                loss_batch += loss_fn(low_res_masks, masks[i].unsqueeze(0))

            loss = loss_batch / images.shape[0]
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

    torch.save({"model": sam3.state_dict()}, output_weights)
    return output_weights


**SAM 3 ENTRENAMIENTO**

En primer lugar, se definen las rutas a los distintos recursos que se van a usar: el dataset, el fichero de cajas delimitadoras, los pesos preentrenados y las rutas donde se guardarán los resultados y los pesos del modelo entrenado. En este caso, se procede a evaluar el rendimiento del modelo SAM 3.

Respecto a la función de evaluación, esta función evalúa el modelo fine-tuneado sobre el conjunto de test. A diferencia de los modelos anteriores, los pesos entrenados se cargan manualmente extrayendo el modelo interno del wrapper de Ultralytics y aplicando el state dict guardado durante el entrenamiento. Para cada imagen se calcula el punto central a partir de la caja delimitadora y se realiza la inferencia, calculando todas las métricas sobre la máscara predicha. Al final se devuelve un diccionario con la media de cada métrica sobre todo el conjunto de test.

Finalmente, se lanza el entrenamiento midiendo el tiempo que tarda, y después se evalúa el modelo resultante, midiendo también su tiempo. Ambos tiempos se añaden al diccionario de resultados, que finalmente se guarda en un CSV. Si el fichero ya existe se añade una nueva línea sin sobreescribir los resultados anteriores, lo que permite ir acumulando los resultados de distintos experimentos en el mismo fichero.

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG"
bboxes = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG\\kavsir_bboxes.json"
weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam3.pt"
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"
output_weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_sam3_kvasir.pt"


def evaluate_model(dataset, weights_path):
    with open(os.path.join(dataset, "kavsir_bboxes.json")) as f:
        bboxes = json.load(f)

    sam3_wrapper = SAM(weights)
    sam3 = sam3_wrapper.model
    for param in sam3.parameters():
        param.data = param.data.to(device)
    for buffer in sam3.buffers():
        buffer.data = buffer.data.to(device)
    sd = torch.load(weights_path)["model"]
    sam3.load_state_dict(sd)
    sam3.eval()

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    for img_name, info in bboxes.items():
        img_path  = os.path.join(dataset, "images", "test", img_name + ".jpg")
        mask_path = os.path.join(dataset, "masks",  "test", img_name + ".jpg")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        bbox_info = info["bbox"][0]

        xmin = bbox_info["xmin"]
        ymin = bbox_info["ymin"]
        xmax = bbox_info["xmax"]
        ymax = bbox_info["ymax"]

        w = xmax - xmin
        h = ymax - ymin

        central_point = [[xmin + w / 2, ymin + h / 2]]

        results, latency, vram = measure_inference_central_point(sam3_wrapper, img_path, np.array(central_point))

        if results is None or results[0].masks is None:
            continue
        scores = results[0].boxes.conf.cpu().numpy()
        best_idx  = np.argmax(scores)
        pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam_3_kvasir"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results


start_train = time.time()
trained_weights = train_sam(dataset, bboxes, weights, output_weights)
train_time = time.time() - start_train

start_eval = time.time()
results = evaluate_model(dataset, trained_weights)
eval_time = time.time() - start_eval

results["train_time_minutes"] = [round(train_time / 60, 2)]
results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


c:\Users\DanielTalavera\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ultralytics 8.4.26  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)


C:\Users\DanielTalavera\AppData\Local\Temp\ipykernel_896\3491141125.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(weights_path)["model"]


WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must